# TP 2 - Elos

**Objectif** : Implémenter K-Means en 1D avec des np.ndarray à la place des listes Python

## 1. Import des bibliothèques

In [11]:
import numpy as np

## 2. Etape 1 : Affectation des points au centroïde le plus proche
Pour chaque point, on calcule sa distance à chaque centroïde, 
puis on choisit le centroïde le plus proche.

Entrées :
- data_points_1d : ndarray de forme (n,)  (ex : [2, 3, 10])
- centroids_1d   : ndarray de forme (k,)  (ex : [2, 10])

Sortie :
- labels : ndarray de forme (n,) contenant l'indice du cluster (0..k-1)

Remarque : Même si les données sont 1D, la table des distances est 2D : (n, k).

### a. Transformer data_points_1d (n,) en colonne (n, 1)
Exemple : [2, 3, 10] devient [[2], [3], [10]]

In [12]:
def transform_1d_to_column(data_points_1d):
    points_as_column = data_points_1d[:, None]
    return points_as_column

### b. Transformer centroids_1d (k,) en ligne (1, k)
Exemple : [2, 10] devient [[2, 10]]

In [13]:
def transform_1d_to_row(centroids_1d):
    centroids_as_row = centroids_1d[None, :]
    return centroids_as_row


### c. Calculer toutes les distances (nombre positif ou négatif) de chaque point vers tous les centroïdes
Chaque point est soustrait à chaque centroïde

In [14]:
def calculate_signed_distances_points_centroids(points_as_column, centroids_as_row):
    differences = points_as_column - centroids_as_row
    return differences


### d. Rendre les distances positives
Calculer la valeur absolue.

In [15]:
def calculate_positive_distances_from_signed_differences(signed_distances):
    distances = np.abs(signed_distances)
    return distances


### e. Trouver le centroïde le plus proche de chaque point
Pour chaque point (chaque ligne), on prend la distance minimale

Indice : argmin renvoie l'indice du centroïde le plus proche

In [16]:
def assign_points_to_nearest_centroid(distance_matrix):
    labels = np.argmin(distance_matrix, axis=1)
    return labels


### f. Assembler les fonctions de l'étape 1

In [17]:
def step1_assign_points_to_clusters(data_points_1d, centroids_1d):
    points_as_column = transform_1d_to_column(data_points_1d)          # (n, 1)
    centroids_as_row = transform_1d_to_row(centroids_1d)               # (1, k)

    signed_distances = calculate_signed_distances_points_centroids(
        points_as_column, centroids_as_row
    )                                                                  # (n, k)

    distances_matrix = calculate_positive_distances_from_signed_differences(
        signed_distances
    )                                                                  # (n, k)

    labels = assign_points_to_nearest_centroid(distances_matrix)  # (n,)

    return labels, distances_matrix


## Etape 2 : Placer les centroïdes au milieu de leur cluster 

Pour chaque cluster, on calcule la position du centroïde : la moyenne des points du cluster.

Entrées :
- data_points_1d : ndarray (n,)
- labels : ndarray (n,) : cluster de chaque point
- number_of_clusters : int (k)

Sortie :
- new_centroids_1d : ndarray (k,)

Cas spécial : cluster vide
- Si aucun point n'est assigné à un cluster, on ne peut pas calculer une moyenne.
- Ici, on met np.nan.

In [18]:
def step2_recompute_centroids(data_points_1d, labels, number_of_clusters, verbose=True):
    new_centroids_1d = np.empty(number_of_clusters, dtype=float)

    for cluster_id in range(number_of_clusters):
        # Récupérer les points affectés à ce cluster 
        # avec un masque booléen : True pour les points dont labels == cluster_id
        mask = (labels == cluster_id)
        points_in_cluster = data_points_1d[mask]

        # Calculer la moyenne (si le cluster n'est pas vide)
        if points_in_cluster.size > 0:
            new_centroids_1d[cluster_id] = points_in_cluster.mean()
        else:
            new_centroids_1d[cluster_id] = np.nan
    return new_centroids_1d


## K-Means complet

Boucle :
1) Affecter les points au centroïde le plus proche
2) Recalculer les centroïdes (moyennes)
3) Recommencer jusqu'à convergence (quand les labels ne changent plus)

In [19]:
def kmeans_1d_numpy(data_points_1d, initial_centroids_1d, max_iterations=10):
    # On convertit en float : les moyennes (centroïdes) peuvent être décimales
    data_points_1d = data_points_1d.astype(float)
    centroids_1d = initial_centroids_1d.astype(float)

    number_of_clusters = centroids_1d.shape[0]
    previous_labels = None

    for iteration in range(max_iterations):
        # Étape 1 : Affectation
        labels, _distance_matrix = step1_assign_points_to_clusters(
            data_points_1d, centroids_1d
        )
        
        # Convergence : si l'affectation ne change plus, on arrête
        if previous_labels is not None and np.array_equal(labels, previous_labels):
            break
        previous_labels = labels

        # Étape 2 : Mise à jour des centroïdes
        centroids_1d = step2_recompute_centroids(
            data_points_1d, labels, number_of_clusters
        )

    return centroids_1d, previous_labels


## Initialisation

In [20]:
data_points_1d = np.array([2, 3, 10], dtype=float)

# Nombre et positions initiales des clusters (choisies)
initial_centroids_1d = np.array([2.0, 10.0], dtype=float)

final_centroids, final_labels = kmeans_1d_numpy(
    data_points_1d=data_points_1d,
    initial_centroids_1d=initial_centroids_1d,
    max_iterations=10

)

print(final_centroids, final_labels)


[ 2.5 10. ] [0 0 1]
